# ML-06 — Model for the Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ahmedosrf/flyrank-ml-internship-ahmedosrf/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

## 1. Method choice and why

I keep Lane 2 locked: **content-refresh opportunity scoring**. The task is a ranking problem: the analyst needs a short list of pages to review first, not an automatic edit.

I use **Logistic Regression** as the first learned model. It produces a probability-like score for ranking, is reproducible and readable, and gives a useful coefficient-based sanity check. A more complex model is not justified unless it earns a clear improvement over the Week-4 rule on the same split and metric. The target is an evaluation proxy derived from `trend_direction == 'down'`; it is used only as the outcome, never as an input feature.

The decision metric is **Precision@10 and Precision@50**, because the action is to inspect the first small queue of pages. I also report average precision and ROC AUC as secondary diagnostics, not as substitutes for the operational ranking metric.

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.model_selection import GroupShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score

REPO = Path('/home/ubuntu/flyrank-ml-internship-starter')
DATA = REPO / 'data/raw/content_refresh_anonymized.csv'
OUT_DIR = REPO / 'work/outputs'
OUT_DIR.mkdir(parents=True, exist_ok=True)
SEED = 42

df = pd.read_csv(DATA)
df['target'] = (df['trend_direction'] == 'down').astype(int)

# These are pre-snapshot or current-window signals. Label-derived trend fields are excluded.
NUMERIC_FEATURES = [
    'impressions_90d', 'clicks_90d', 'sessions_90d', 'ctr', 'avg_position',
    'engagement_rate', 'content_age_days', 'days_since_last_update',
    'word_count', 'search_volume', 'cpc'
]
CATEGORICAL_FEATURES = [
    'competition_level', 'content_type', 'main_intent', 'age_tier',
    'freshness_tier', 'word_count_tier', 'impression_tier', 'position_tier'
]
MODEL_FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES
FORBIDDEN = {'trend_direction', 'trend_pct', 'is_declining_label', 'target'}
assert not (set(MODEL_FEATURES) & FORBIDDEN)
assert set(MODEL_FEATURES).issubset(df.columns)
print('Rows:', len(df), '| clients:', df['client_id'].nunique())
print('Target rate:', round(df['target'].mean(), 4))
print('Model features:', len(MODEL_FEATURES), '| forbidden overlap:', set(MODEL_FEATURES) & FORBIDDEN)

Rows: 30000 | clients: 32
Target rate: 0.5421
Model features: 19 | forbidden overlap: set()


## 2. Split design

I use one **grouped holdout split by `client_id`** with a fixed seed of 42: 80% of rows for training and 20% for testing. A client cannot appear in both sets, which is stricter than a random row split and reduces the risk that client-specific patterns make the score look better than it will on an unseen client. The baseline is calculated on the exact same test rows and with the exact same Precision@K function.

In [2]:
X = df[MODEL_FEATURES]
y = df['target']
groups = df['client_id']
gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=SEED)
train_idx, test_idx = next(gss.split(X, y, groups=groups))
train = df.iloc[train_idx].copy()
test = df.iloc[test_idx].copy()
assert set(train['client_id']).isdisjoint(set(test['client_id']))
print('Train rows:', len(train), '| test rows:', len(test))
print('Train clients:', train['client_id'].nunique(), '| test clients:', test['client_id'].nunique())
print('Client overlap:', len(set(train['client_id']) & set(test['client_id'])))
display(pd.DataFrame({'split':['train','test'], 'rows':[len(train),len(test)], 'clients':[train.client_id.nunique(),test.client_id.nunique()], 'target_rate':[train.target.mean(),test.target.mean()]}))

Train rows: 23837 | test rows: 6163
Train clients: 25 | test clients: 7


Client overlap: 0


,split,rows,clients,target_rate
0,train,23837,25,0.550111
1,test,6163,7,0.510952


## 3. Train + compare vs my baseline

The preprocessing imputes numeric missing values with the training median and categorical missing values with the training mode, then one-hot encodes categories. The model is fitted only on the training clients. The Week-4 rule is reproduced below rather than copied as a saved result, so baseline and model are measured on the same held-out clients.

In [3]:
preprocess = ColumnTransformer([
    ('numeric', Pipeline([
        ('impute', SimpleImputer(strategy='median')),
        ('scale', StandardScaler())
    ]), NUMERIC_FEATURES),
    ('categorical', Pipeline([
        ('impute', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore'))
    ]), CATEGORICAL_FEATURES),
])
model = Pipeline([
    ('preprocess', preprocess),
    ('classifier', LogisticRegression(max_iter=1000, solver='liblinear', random_state=SEED))
])
model.fit(train[MODEL_FEATURES], train['target'])
model_score = model.predict_proba(test[MODEL_FEATURES])[:, 1]

# Week-4 baseline rule, recomputed on the same test rows.
visible = (test['impressions_90d'] >= 500).astype(int)
page_one_two = ((test['avg_position'] > 0) & (test['avg_position'] <= 20)).astype(int)
low_ctr = (test['ctr'] < 0.5).astype(int)
baseline_score = visible * page_one_two * low_ctr * np.log1p(test['impressions_90d'].astype(float))

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores), kind='mergesort')[:k]
    return float(np.asarray(labels)[order].mean())

rows=[]
for name, scores in [('Week-4 rule baseline', baseline_score), ('Logistic Regression', model_score)]:
    rows.append({
        'method': name,
        'precision_at_10': precision_at_k(scores, test['target'], 10),
        'precision_at_50': precision_at_k(scores, test['target'], 50),
        'precision_at_100': precision_at_k(scores, test['target'], 100),
        'average_precision': average_precision_score(test['target'], scores),
        'roc_auc': roc_auc_score(test['target'], scores),
    })
comparison = pd.DataFrame(rows)
comparison['test_base_rate'] = test['target'].mean()
display(comparison.round(4))
metrics = {
    'seed': SEED,
    'split': 'GroupShuffleSplit test_size=0.20 by client_id',
    'train_rows': int(len(train)), 'test_rows': int(len(test)),
    'train_clients': int(train.client_id.nunique()), 'test_clients': int(test.client_id.nunique()),
    'baseline': rows[0], 'model': rows[1], 'test_base_rate': float(test.target.mean()),
    'features': MODEL_FEATURES,
}
(OUT_DIR / 'ml06_model_metrics.json').write_text(json.dumps(metrics, indent=2) + '\n')

,method,precision_at_10,precision_at_50,precision_at_100,average_precision,roc_auc,test_base_rate
0,Week-4 rule baseline,0.4,0.42,0.44,0.4991,0.5118,0.511
1,Logistic Regression,0.8,0.58,0.48,0.5386,0.5503,0.511


1058

## 4. Errors and interpretation

The table below shows the largest absolute logistic coefficients after preprocessing. These are directional associations inside this split, not causal effects. I also show three concrete threshold errors. A false positive means the model sent a page to the review queue even though the evaluation proxy was not down; a false negative means it missed a down-trending page. These cases are why the score remains decision support for a human editor rather than an automatic content change.

In [4]:
feature_names = model.named_steps['preprocess'].get_feature_names_out()
coefficients = model.named_steps['classifier'].coef_[0]
importance = pd.DataFrame({'feature': feature_names, 'coefficient': coefficients})
importance['abs_coefficient'] = importance['coefficient'].abs()
print('Top coefficient signals:')
display(importance.sort_values('abs_coefficient', ascending=False).head(10).drop(columns='abs_coefficient').round(4))

error_view = test[['content_id','client_id','target'] + NUMERIC_FEATURES].copy()
error_view['model_score'] = model_score
error_view['predicted_at_0_5'] = (model_score >= 0.5).astype(int)
error_view['error_type'] = np.where(
    (error_view['predicted_at_0_5'] == 1) & (error_view['target'] == 0), 'false_positive',
    np.where((error_view['predicted_at_0_5'] == 0) & (error_view['target'] == 1), 'false_negative', 'correct')
)
errors = error_view[error_view['error_type'] != 'correct'].copy()
print('Threshold errors:', len(errors), '| false positives:', int((errors.error_type=='false_positive').sum()), '| false negatives:', int((errors.error_type=='false_negative').sum()))
print('Three concrete errors:')
display(errors.head(3)[['content_id','client_id','error_type','target','model_score','impressions_90d','ctr','avg_position','content_age_days']].round(4))
print('Interpretation: the model improves the short ranking cutoffs in this grouped holdout, but its ROC AUC is only modestly above chance. The largest signals are position, freshness, and content structure; they are plausible prioritization cues, not proof that editing them will cause a recovery. The error rows show that low-volume pages and pages with mixed engagement signals remain difficult.')

Top coefficient signals:


,feature,coefficient
41,categorical__position_tier_top_3,-1.3038
26,categorical__freshness_tier_181+,-0.4708
29,categorical__word_count_tier_1000-2000,0.4625
15,categorical__content_type_feedly article,-0.4519
39,categorical__position_tier_page_3_5,0.4221
40,categorical__position_tier_striking,0.3991
19,categorical__main_intent_navigational,-0.3443
6,numeric__content_age_days,-0.3287
35,categorical__impression_tier_low,-0.2762
22,categorical__age_tier_31-90,0.2733


Threshold errors: 2789 | false positives: 1617 | false negatives: 1172
Three concrete errors:


,content_id,client_id,error_type,target,model_score,impressions_90d,ctr,avg_position,content_age_days
13,content_a5a2fbc76336,client_8527a891e2,false_positive,0,0.7434,307,0.00,39.8,238
19,content_af865035b328,client_f369cb89fc,false_negative,1,0.4635,99,2.02,6.9,187
23,content_2da6ae9d0882,client_e629fa6598,false_negative,1,0.3920,297,0.34,13.9,502


Interpretation: the model improves the short ranking cutoffs in this grouped holdout, but its ROC AUC is only modestly above chance. The largest signals are position, freshness, and content structure; they are plausible prioritization cues, not proof that editing them will cause a recovery. The error rows show that low-volume pages and pages with mixed engagement signals remain difficult.


## Self-check

- [x] The method fits a ranking lane and produces scores rather than automatic edits.
- [x] Baseline and model use the same held-out client groups and the same Precision@K functions.
- [x] The notebook reports Precision@10, Precision@50, Precision@100, average precision, ROC AUC, and the test base rate.
- [x] The top coefficient signals and three concrete errors are shown with cautious interpretation.
- [x] No label-derived trend field or future-window field is in `MODEL_FEATURES`.
- [x] The seed and split design are recorded so the run is reproducible.
